# Thí nghiệm 5: So sánh TinyCAE INT8 và JPEG trên ESP32

## Mục tiêu

Thí nghiệm tổng hợp kết quả TinyCAE INT8 và JPEG trên cùng 20 ảnh hardware test. Các tiêu chí gồm kích thước dữ liệu nén, BPP, tỷ lệ nén, PSNR, SSIM và thời gian xử lý lõi trên ESP32. Kết quả được trình bày dưới dạng bảng và biểu đồ rate–distortion–latency.

## 1. Thu thập và kiểm tra dữ liệu

Dữ liệu đầu vào gồm CSV TinyCAE ESP32 và bốn CSV JPEG tại quality 30, 50, 70 và 90. Danh sách `image_id` và thứ tự mẫu phải giống nhau giữa mọi codec để bảo đảm so sánh ghép cặp trên cùng nội dung ảnh.

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
FINAL_CSV = RESULTS_DIR / 'final_codec_comparison.csv'
TINYCAE_PATH = RESULTS_DIR / 'tinycae_esp32_20.csv'
JPEG_QUALITIES = [30, 50, 70, 90]

def read_csv(path):
    with path.open(newline='', encoding='utf-8') as stream:
        return list(csv.DictReader(stream))

tinycae = read_csv(TINYCAE_PATH)
jpeg = {quality: read_csv(RESULTS_DIR / f'cifar10_jpeg_q{quality}.csv') for quality in JPEG_QUALITIES}
assert len(tinycae) == 20
assert all(len(rows) == 20 for rows in jpeg.values())
tinycae_ids = [row['image_id'] for row in tinycae]
assert all([row['image_id'] for row in rows] == tinycae_ids for rows in jpeg.values())
print('Đã xác minh tất cả codec dùng cùng 20 ảnh, cùng thứ tự.')

## 2. Chỉ số tổng hợp

TinyCAE sử dụng latent cố định 1.024 byte; JPEG có kích thước bitstream phụ thuộc nội dung và quality. BPP được tính trên 1.024 pixel. Tỷ lệ nén lấy kích thước ảnh RGB888 3.072 byte chia kích thước biểu diễn nén. Thời gian là thời gian lõi `Invoke()` hoặc JPEG encode trên ESP32, không bao gồm truyền UART. Trung bình và độ lệch chuẩn được tính trên 20 ảnh.

In [ ]:
def summarize(name, sizes, bpp, ratios, times, psnr, ssim):
    return {
        'codec': name,
        'images': len(sizes),
        'mean_bytes': float(np.mean(sizes)),
        'mean_bpp': float(np.mean(bpp)),
        'mean_compression_ratio': float(np.mean(ratios)),
        'mean_time_us': float(np.mean(times)),
        'std_time_us': float(np.std(times)),
        'mean_psnr_db': float(np.mean(psnr)),
        'std_psnr_db': float(np.std(psnr)),
        'mean_ssim': float(np.mean(ssim)),
        'std_ssim': float(np.std(ssim)),
    }

tiny_sizes = np.asarray([float(row['latent_bytes']) for row in tinycae])
summary_rows = [summarize(
    'TinyCAE INT8',
    tiny_sizes,
    tiny_sizes * 8 / (32 * 32),
    3072 / tiny_sizes,
    [float(row['invoke_time_us']) for row in tinycae],
    [float(row['esp32_psnr_db']) for row in tinycae],
    [float(row['esp32_ssim']) for row in tinycae],
)]
for quality in JPEG_QUALITIES:
    rows = jpeg[quality]
    summary_rows.append(summarize(
        f'JPEG Q{quality}',
        [float(row['jpeg_bytes']) for row in rows],
        [float(row['bits_per_pixel']) for row in rows],
        [float(row['compression_ratio']) for row in rows],
        [float(row['encode_time_us']) for row in rows],
        [float(row['psnr_db']) for row in rows],
        [float(row['ssim']) for row in rows],
    ))

with FINAL_CSV.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=summary_rows[0].keys())
    writer.writeheader()
    writer.writerows(summary_rows)

for row in summary_rows:
    print(
        f"{row['codec']:12s} | {row['mean_bytes']:7.2f} B | {row['mean_bpp']:.4f} bpp | "
        f"PSNR {row['mean_psnr_db']:.4f} dB | SSIM {row['mean_ssim']:.6f} | "
        f"time {row['mean_time_us'] / 1000:.3f} ms"
    )
print('Đã lưu:', FINAL_CSV)

## 3. So sánh ghép cặp với JPEG Q70

JPEG Q70 có BPP trung bình gần BPP 8,0 của TinyCAE nhất trong bốn cấu hình khảo sát. Chênh lệch PSNR, SSIM và thời gian được tính theo từng cặp ảnh trước khi lấy trung bình, tránh sai lệch do phân bố nội dung giữa các tập khác nhau.

In [ ]:
jpeg_q70 = jpeg[70]
delta_psnr = np.asarray([float(t['esp32_psnr_db']) - float(j['psnr_db']) for t, j in zip(tinycae, jpeg_q70)])
delta_ssim = np.asarray([float(t['esp32_ssim']) - float(j['ssim']) for t, j in zip(tinycae, jpeg_q70)])
delta_time_ms = np.asarray([float(t['invoke_time_us']) - float(j['encode_time_us']) for t, j in zip(tinycae, jpeg_q70)]) / 1000.0
print(f'TinyCAE - JPEG Q70 PSNR: {delta_psnr.mean():.6f} dB')
print(f'TinyCAE - JPEG Q70 SSIM: {delta_ssim.mean():.8f}')
print(f'TinyCAE - JPEG Q70 thời gian: {delta_time_ms.mean():.3f} ms')
print(f'TinyCAE PSNR cao hơn Q70 ở {int(np.sum(delta_psnr > 0))}/20 ảnh')
print(f'TinyCAE SSIM cao hơn Q70 ở {int(np.sum(delta_ssim > 0))}/20 ảnh')

## 4. Biểu diễn kết quả

Hai biểu đồ phân tán BPP–PSNR và BPP–SSIM mô tả quan hệ giữa bitrate và chất lượng. Biểu đồ cột thời gian thể hiện chênh lệch chi phí tính toán trên ESP32. TinyCAE được ký hiệu riêng với nhóm JPEG để làm rõ hai họ codec.

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
names = [row['codec'] for row in summary_rows]
bpp = [row['mean_bpp'] for row in summary_rows]
psnr = [row['mean_psnr_db'] for row in summary_rows]
ssim = [row['mean_ssim'] for row in summary_rows]
time_ms = [row['mean_time_us'] / 1000.0 for row in summary_rows]
colors = ['tab:red'] + ['tab:blue'] * 4

def scatter_metric(y, ylabel, title, filename):
    plt.figure(figsize=(7, 5))
    for index, name in enumerate(names):
        plt.scatter(bpp[index], y[index], s=90, color=colors[index], label=name)
        plt.annotate(name, (bpp[index], y[index]), xytext=(5, 5), textcoords='offset points')
    plt.xlabel('Bits per pixel (BPP)')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=180)
    plt.show()

scatter_metric(psnr, 'PSNR (dB)', 'BPP–PSNR trên 20 ảnh ESP32', 'bpp_psnr.png')
scatter_metric(ssim, 'SSIM', 'BPP–SSIM trên 20 ảnh ESP32', 'bpp_ssim.png')

plt.figure(figsize=(8, 5))
bars = plt.bar(names, time_ms, color=colors)
plt.ylabel('Thời gian lõi trên ESP32 (ms)')
plt.title('Thời gian TinyCAE Invoke và JPEG encode')
plt.xticks(rotation=25, ha='right')
plt.grid(axis='y', alpha=0.3)
for bar, value in zip(bars, time_ms):
    plt.text(bar.get_x() + bar.get_width()/2, value, f'{value:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'latency_comparison.png', dpi=180)
plt.show()
print('Biểu đồ:', FIGURES_DIR)

## 5. Nguyên tắc thảo luận

Kết luận được xây dựng từ đồng thời bitrate, PSNR, SSIM, thời gian, kích thước model và bộ nhớ; không sử dụng một chỉ số riêng lẻ. TinyCAE cung cấp latent kích thước cố định và chứng minh khả năng triển khai encoder học sâu INT8 trên ESP32. JPEG cung cấp tốc độ cao, quality điều chỉnh được và hiệu quả rate–distortion tốt trong cấu hình khảo sát.